# M2 — Watermarks: scenario testing (v2)

M2-only isolation: every verdict here is caused exclusively by watermark
evidence — M1/M3/M4 are silenced.

**Three active decoders** (what survived evaluation):

| # | scheme | detects | type | verdict tier |
|---|---|---|---|---|
| 1 | dwtDct | SD 1.x/2.x/SDXL (invisible-watermark) | payload match | verified |
| 2 | trustmark-P/Q/B | Adobe Content Authenticity outputs | payload decode, all variants | verified |
| 3 | stable-signature-bzh | SDXL-turbo IMATAG builds | zero-bit p-value | likely |

**Rejected after evaluation — SynthID CNN surrogate.** A third-party ensemble
(claimed 0.97 acc) fired on *every* image we tested: AI outputs, personal
photos, synthetic fixtures (P=0.65–1.00). Shortcut learning, unaudited repo.
Removed from the active registry; `synthid_cnn.py` kept as a cautionary tale.

**SynthID reality (2026):** independent detection is cryptographically
impossible without Google's private keys. Official paths are oracle-only
(SynthID Detector portal, Gemini app) or partner-preview (Content Detection
API on Google Cloud). M1 reads SynthID *declarations* from C2PA manifests
(`c2pa.watermarked.unbound`) — declared, not independently verified.

Convention: commit WITH outputs — run record.

In [1]:
# ── Setup — single cell, no restart ────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .

# --no-deps: trustmark's numpy<2 pin is defensive metadata — it runs fine on
# Colab's numpy 2.x (verified), and letting pip "fix" numpy corrupts the install.
# Its real runtime deps (torch, torchvision, pillow) are already in Colab.
%pip install -q --no-deps trustmark 2>/dev/null || echo "trustmark not available"
%pip install -q transformers 2>/dev/null || echo "transformers not available"

import numpy as np
from pathlib import Path
from PIL import Image
from ai_image_id.ingest import ingest
from ai_image_id.watermark import analyze_watermarks, dwt_dct, SDXL_BITS, SD_V1_BITS
from ai_image_id.fusion import fuse
from ai_image_id.schema import Evidence, ProvenanceEvidence, SpectrumEvidence, Verdict

def report(path, label=""):
    """M2-only evidence card. No provenance, no FFT, no detector."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    evidence = Evidence(provenance=ProvenanceEvidence(), watermarks=wms,
                        spectrum=SpectrumEvidence(valid=False), detector=None)
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    print(f"┌─ {label or path}")
    print(f"│ verdict: {r.ai_verdict.value} ({r.confidence})")
    for w in wms:
        status = "DETECTED" if w.detected else ("n/a" if not w.applicable else "not detected")
        detail = f" acc/p={w.bit_accuracy}" if w.bit_accuracy is not None else ""
        detail += f" payload={w.matched_payload}" if w.matched_payload else ""
        print(f"│   {w.scheme:<22} {status}{detail}")
        if w.notes and not w.applicable: print(f"│     └ {w.notes[:90]}")
    print(f"└ notes: {r.notes}\n")
    return r

def wm_row(path):
    """One-line per-scheme summary for matrix tables."""
    img = ingest(path)
    wms = analyze_watermarks(img.rgb)
    evidence = Evidence(provenance=ProvenanceEvidence(), watermarks=wms,
                        spectrum=SpectrumEvidence(valid=False), detector=None)
    r = fuse(evidence, sha256=img.sha256, phash=img.phash)
    cols = {w.scheme: (w.detected if w.applicable else None)
            for w in wms if w.scheme not in ("synthid", "meta-invisible")}
    return cols, r.ai_verdict.value

# Availability — import checks only, NO model downloads (those happen lazily
# on the first real report() call, where a pause is expected)
print("decoder availability:")
print(f"  {'dwtDct':<22} ✓ ready (vendored)")
for name, mod in [("trustmark P/Q/B", "trustmark"), ("stable-signature-bzh", "transformers")]:
    try:
        __import__(mod); print(f"  {name:<22} ✓ ready")
    except ImportError: print(f"  {name:<22} ✗ {mod} not installed")

already cloned
/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ai-image-id (pyproject.toml) ... done
decoder availability:
  dwtDct                 ✓ ready (vendored)
  trustmark P/Q/B        ✓ ready
  stable-signature-bzh   ✓ ready


## 1 — DWT-DCT: round-trip + fragility (established results)

Prior measurements: lossless round-trip ≈0.92 (barely above the 0.90
threshold), **any** JPEG (even Q92) → ≈0.51 = coin flip, indistinguishable
from an unwatermarked image. Re-run to confirm on this session, then the
sweep chart. Conclusion stands: detects only pristine, never-recompressed
SD PNGs — near-zero real-world reach.

In [2]:
FIX = Path("/content/fixtures_m2"); FIX.mkdir(exist_ok=True)
rng = np.random.default_rng(7)
y, x = np.mgrid[0:512, 0:512]
base = np.clip(np.stack([120+60*np.sin(x/97), 100+50*np.cos(y/83),
                         90+40*np.sin((x+y)/71)], -1) + rng.normal(0,6,(512,512,3)),
               0, 255).astype(np.uint8)
marked = dwt_dct.embed(base, SDXL_BITS)

p = FIX/"sdxl_lossless.png"; Image.fromarray(marked).save(p)
report(str(p), "SDXL payload, lossless PNG")
p = FIX/"sdxl_q92.jpg"; Image.fromarray(marked).save(p, quality=92)
report(str(p), "SDXL payload, Q92 (expect coin flip)")
p = FIX/"clean.png"; Image.fromarray(base).save(p)
report(str(p), "no watermark (baseline)")

Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/trustmark_P.yaml
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/decoder_P.ckpt
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/encoder_P.ckpt
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/trustmark_rm_P.yaml
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/trustmark_rm_P.ckpt
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/trustmark_Q.yaml
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/decoder_Q.ckpt
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/encoder_Q.ckpt
Fetching model file (once only): /usr/local/lib/python3.12/dist-packages/trustmark/models/trustmark_rm_Q.yaml
Fetching model file (once only): /us

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 44.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

┌─ SDXL payload, lossless PNG
│ verdict: verified (0.95)
│   dwtDct                 DETECTED acc/p=0.917 payload=sdxl-48bit
│   trustmark              not detected
│   stable-signature-bzh   not detected acc/p=0.0
│   synthid                n/a
│     └ independent detection impossible (needs Google's keys); CNN surrogate evaluated and reject
│   meta-invisible         n/a
│     └ no public decoder; Meta's proprietary scheme
└ notes: ["SD invisible watermark matched 'sdxl-48bit' (bit accuracy 0.917)"]

┌─ SDXL payload, Q92 (expect coin flip)
│ verdict: inconclusive (0.5)
│   dwtDct                 not detected acc/p=0.507
│   trustmark              not detected
│   stable-signature-bzh   not detected acc/p=0.0
│   synthid                n/a
│     └ independent detection impossible (needs Google's keys); CNN surrogate evaluated and reject
│   meta-invisible         n/a
│     └ no public decoder; Meta's proprietary scheme
└ notes: ['no AI-indicative signal found', 'absence of watermarks/m

AnalysisResult(ai_verdict=<Verdict.INCONCLUSIVE: 'inconclusive'>, confidence=0.5, evidence=Evidence(provenance=ProvenanceEvidence(c2pa_present=False, c2pa_valid=None, c2pa_signature_valid=None, c2pa_signer_trusted=None, c2pa_generator=None, c2pa_capture_claim=False, c2pa_actions=[], c2pa_raw=None, generation_params_tool=None, generation_params_model=None, iptc_digital_source_type=None, iptc_source_category=None, software=None, camera_exif_present=False, camera_exif_fields=0, ai_metadata_hits=[]), watermarks=[WatermarkEvidence(scheme='dwtDct', applicable=True, detected=False, matched_payload=None, bit_accuracy=0.542, notes=None), WatermarkEvidence(scheme='trustmark', applicable=True, detected=False, matched_payload=None, bit_accuracy=None, notes='no TrustMark watermark found (tried P/Q/B)'), WatermarkEvidence(scheme='stable-signature-bzh', applicable=True, detected=False, matched_payload=None, bit_accuracy=0.0, notes='not detected, p(watermarked)=0.0'), WatermarkEvidence(scheme='synthid

## 2 — TrustMark: round-trip + the variant lesson

The decoder loops P/Q/B because variants are mutually incompatible — we
learned this the measured way: Adobe's Content Authenticity app embeds with
**P** (manifest soft-binding `alg=com.adobe.trustmark.P`), while our first
integration only tried Q and reported false negatives on a provably
watermarked file. Round-trip below embeds with Q; §3 tests the real
Adobe-P image.

In [ ]:
from trustmark import TrustMark
tm_q = TrustMark(verbose=False, model_type='Q')

img = Image.fromarray(base)
encoded = tm_q.encode(img, "TESTMARK0")
enc_img = encoded[0] if isinstance(encoded, tuple) else encoded
enc_img.save("/content/tm_encoded.png")
print("in-memory decode:", tm_q.decode(enc_img))
print("after save+reload:", tm_q.decode(Image.open("/content/tm_encoded.png").convert("RGB")))

# And through OUR module (must find it via the variant loop):
report("/content/tm_encoded.png", "TrustMark-Q self-encoded")

## 3 — Real images: the gauntlet

Upload and check against expectations:

| image | dwtDct | trustmark | bzh | note |
|---|---|---|---|---|
| Content Authenticity app output (your applied Firefly) | · | **✓ trustmark-P** | · | THE validation target |
| Raw Firefly export | · | · | · | C2PA only — no watermark embedded (measured) |
| OpenAI/ChatGPT PNG | · | · | · | SynthID undetectable; M1 reads the declaration |
| Real SD PNG (pristine, watermark enabled) | ✓? | · | · | if you can source one |
| Phone photo | · | · | · | negative control |

In [ ]:
from google.colab import files
up = files.upload()
for name in sorted(up):
    report(name)

## 4 — Transport-degradation matrix

Same seven hops as notebook 03 (M1). Upload the **Content Authenticity app
output** — it's the only image we have carrying a decodable watermark, so
it's the only one whose survival curve is measurable. The question this
answers: does TrustMark survive the transports that kill C2PA?
(Adobe's design claim: yes — that's the whole point of "durable".)

In [ ]:
import shutil, subprocess
from google.colab import files

up = files.upload()
orig = Path(next(iter(up)))
HOP = Path("/content/hops_m2"); HOP.mkdir(exist_ok=True)
im = Image.open(orig).convert("RGB")

hops = {"0-original": orig}
p = HOP/"1-resave.jpg";     im.save(p, quality=92);  hops["1-resave-jpg"] = p
p = HOP/"2-screenshot.png"; im.save(p);               hops["2-screenshot"] = p
p = HOP/"3-messenger.jpg";  im.resize((im.width//2, im.height//2)).save(p, quality=70)
hops["3-messenger"] = p
p = HOP/("4-strip"+orig.suffix); shutil.copy(orig, p)
subprocess.run(["exiftool","-overwrite_original","-all=",str(p)], capture_output=True)
hops["4-exiftool-strip"] = p
p = HOP/"5-crop.jpg"; im.crop((50,50,im.width-50,im.height-50)).save(p, quality=92)
hops["5-crop"] = p
p = HOP/"6-pil-reencode.png"; im.save(p)
hops["6-pil-reencode"] = p

SCHEMES = ["dwtDct", "trustmark", "stable-signature-bzh"]
print(f"{'transport':<20}" + "".join(f"{s:<24}" for s in SCHEMES) + "verdict")
for name, path in hops.items():
    cols, verdict = wm_row(path)
    row = f"{name:<20}"
    for s in SCHEMES:
        # trustmark may report as trustmark-P/Q/B when detected
        hit = next((v for k, v in cols.items() if k.startswith(s)), None)
        row += f"{'✓' if hit else ('·' if hit is False else '—'):<24}"
    print(row + verdict)

### Reading the matrix — the M1 vs M2 composite

M1's measured row (notebook 03): C2PA survives ONLY exiftool-strip.
If TrustMark shows ✓ on re-save / screenshot / crop where C2PA showed ·,
the two modules are **complementary** — metadata proves origin on pristine
files, the watermark carries detection through reprocessing. That composite
is the evidence-fusion thesis as one table, and the blog's key figure.

## 5 — Scope: final M2 coverage map

| generator | scheme | M2 status |
|---|---|---|
| SD 1.x/2.x/SDXL (invisible-watermark on) | DWT-DCT | works; dies at any JPEG (measured) |
| Adobe Content Authenticity outputs | TrustMark P | **works, validated end-to-end** |
| Raw Firefly / firefly.adobe.com | — | no watermark embedded (measured) |
| SDXL-turbo IMATAG builds | Stable Signature | decoder ready; no true-positive sourced yet |
| Google Gemini/Imagen | SynthID | **impossible** without Google's keys; oracle/partner-API only |
| OpenAI GPT-Image-2 | SynthID | same; M1 reads the C2PA declaration instead |
| Meta AI | proprietary | no public decoder |
| Midjourney, Flux, Leonardo, Ideogram… | none | nothing to detect — M1/M4 territory |

**The structural finding:** the watermark ecosystem is closed (Google/OpenAI/
Meta), fragile (DWT-DCT), or opt-in narrow (TrustMark, Stable Signature).
M2 contributes decisive evidence when it fires, but for most wild images the
signal falls to M1 (metadata) and M4 (classifier). The rejected SynthID CNN
is the methodological lesson: third-party checkpoints are unaudited claims
until you evaluate them yourself — ours failed in five uploads.